# 01 — Land source files

**Purpose:** Bring the raw Kitsune SYN DoS files into the landing zone, simulating a source system delivering files to the pipeline.

**Reads from:** `/Volumes/kitsune_project/bronze_layer/raw_files/` (legacy volume)

**Writes to:** `/Volumes/kitsune_dev/raw/landing/`

**Validates:**
- The number of landed files matches the expected list
- Each file has the exact expected size in bytes

If any check fails, the notebook raises an error and stops, so downstream tasks do not run on incomplete data.

**When to run:** Before `02_bronze_ingestion`, every time the pipeline runs.

**Idempotent:** Yes. Files already in landing with the correct size are skipped; only missing or incomplete files are copied.

**Note:** Files are copied from the legacy volume instead of downloaded from UCI, because the full UCI archive is much larger than the SYN DoS subset and Free Edition network access is limited. To run from scratch, download the SYN DoS files from the [UCI ML Repository](https://archive.ics.uci.edu/dataset/516/kitsune+network+attack+dataset) and upload them to the landing volume.

In [0]:
# Source: legacy volume with the original files
SOURCE_DIR = "/Volumes/kitsune_project/bronze_layer/raw_files/"

# Destination: landing zone in the dev catalog
LANDING_DIR = "/Volumes/kitsune_dev/raw/landing/"

# Expected size in bytes of each source file
EXPECTED_SIZES = {
    "SYN DoS_dataset.csv.gz": 2136972919,
    "SYN DoS_labels.csv.gz": 6568025,
}

In [0]:
# Files already in landing: name -> size
already_landed = {}
for f in dbutils.fs.ls(LANDING_DIR):
    already_landed[f.name] = f.size

# Copy only files that are missing or incomplete
for f in dbutils.fs.ls(SOURCE_DIR):
    if already_landed.get(f.name) == f.size:
        print(f"Skipping {f.name}, already landed")
        continue
    print(f"Copying {f.name} ...")
    dbutils.fs.cp(f.path, LANDING_DIR + f.name)

Skipping SYN DoS_dataset.csv.gz, already landed
Skipping SYN DoS_labels.csv.gz, already landed


In [0]:
# Files currently in the landing zone
landed_files = dbutils.fs.ls(LANDING_DIR)

# The number of files must match the expected list
if len(EXPECTED_SIZES) != len(landed_files):
    raise ValueError(f"Expected {len(EXPECTED_SIZES)} files, found {len(landed_files)}")

# Check every file in the landing zone has the expected size
for f in landed_files:
    expected = EXPECTED_SIZES[f.name]
    if f.size != expected:
        raise ValueError(f"{f.name}: expected {expected} bytes, got {f.size}")

print("All files landed with the expected size")

All files landed with the expected size


**Next:** `02_bronze_ingestion`, which loads the landed files into Delta tables in the `bronze` schema.